# cross-entropy-classification-loss — ex1: manual cross-entropy matches F.cross_entropy on logits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-entropy-classification-loss`. Running the final beacon cell reports progress against the `Loss: Cross-entropy classification` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: Cross-entropy classification` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-entropy-classification-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-entropy-classification-loss"
DD_SUBTOPIC = "Loss: Cross-entropy classification"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-entropy classification loss — quick refresher

Cross-entropy for multi-class classification is the negative log-likelihood of the correct class under the model's softmax:

```
p_i        = softmax(logits)[i]            # probability of class i
loss_per_x = -log(p_{y_true})              # NLL for that example
loss       = mean(loss_per_x over batch)   # scalar batch loss
```

PyTorch packages this as `F.cross_entropy(logits, labels)` and computes it in a NUMERICALLY STABLE way via the log-sum-exp trick — you should call it instead of rolling softmax + log + index yourself.

**Critical: `cross_entropy` takes LOGITS, not probabilities.** It applies softmax internally. Passing already-softmaxed values is a common silent bug — the loss still computes, but it's `-log(softmax(softmax(...)))`, which gives near-uniform gradients and the model stops learning.

**Shape contract.** `logits` is `(B, C)`, `labels` is `(B,)` with integer class indices in `[0, C)`. Output is a single scalar (default reduction is `'mean'`).

### Exercise 1 — manual cross-entropy matches F.cross_entropy on logits

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the manual cross-entropy decomposition `loss = -log_softmax(logits)[range(B), labels].mean()` and verify it matches `F.cross_entropy(logits, labels)`.
> Keywords: cross-entropy, softmax, log-softmax, classification
> ```

**KCs targeted:** `cross-entropy-takes-logits-not-probs`, `cross-entropy-equals-mean-of-neg-log-target-probs`

Implement `ex1_manual_cross_entropy(logits, labels)`. Roll your own cross-entropy WITHOUT calling `F.cross_entropy` or `F.nll_loss`. Then we verify it matches PyTorch's reference.

1. `log_probs = F.log_softmax(logits, dim=-1)` — numerically stable log-softmax along the class axis. Shape `(B, C)`.
2. Gather the LOG-PROBABILITY of the CORRECT class for each example. The clearest way: `target_log_probs = log_probs[range(len(labels)), labels]`. Shape `(B,)`.
3. Return `-target_log_probs.mean()` — a scalar tensor.

Inputs:
- `logits`: `(B, C)` raw model outputs.
- `labels`: `(B,)` int64 class indices in `[0, C)`.

Output: scalar loss, must match `F.cross_entropy(logits, labels)` to within 1e-5.

**Critical:** you receive LOGITS, not probabilities. Do NOT softmax-then-log — use `F.log_softmax` to be numerically stable. The reference is `F.cross_entropy` which expects logits.

In [ ]:
import torch.nn.functional as F


def ex1_manual_cross_entropy(logits, labels):
    log_probs = F.log_softmax(logits, dim=-1)
    target_log_probs = log_probs[range(len(labels)), labels]
    return -target_log_probs.mean()


<details><summary>Solution</summary>

```python
import torch.nn.functional as F


def ex1_manual_cross_entropy(logits, labels):
    log_probs = F.log_softmax(logits, dim=-1)
    target_log_probs = log_probs[range(len(labels)), labels]
    return -target_log_probs.mean()
```

**Why `F.log_softmax` not `log(softmax(...))`.** Softmax involves `exp(logits)`, which overflows for large logits (e.g. 1000). `log_softmax` uses the log-sum-exp trick internally — it shifts logits by the per-row max before exponentiating, so the largest exponent is `exp(0) = 1` and the rest are smaller. This is numerically stable for any input magnitude.

**Why `range(len(labels))` for the gather.** `log_probs[range(B), labels]` is fancy-indexing — it picks `log_probs[0, labels[0]], log_probs[1, labels[1]], ...`. Equivalent to `log_probs.gather(1, labels.unsqueeze(1)).squeeze(1)`, just less ceremonial. Both work.

**Why this drill matters even though `F.cross_entropy` exists.** Two reasons: (1) you'll see this exact decomposition in language-modeling losses where you want to weight per-token losses before reducing; (2) `F.cross_entropy(probs, labels)` is the most common 'why isn't my model learning' bug — and you can only diagnose it if you understand what the function expects (LOGITS) vs what you accidentally passed (POST-SOFTMAX PROBS).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()